# Clinical Notes RAG System

A Retrieval-Augmented Generation (RAG) pipeline built on synthetic clinical notes, demonstrating how to extract, chunk, embed, retrieve, and answer questions over an EHR-style document.

## Pipeline overview

| Step | Description |
|------|-------------|
| 1. Data Loading | Extract text from the synthetic clinical PDF |
| 2. Text Cleaning | Strip headers and normalise whitespace |
| 3. Text Chunking | Three-pass chunking (section → sub-section → merge) |
| 4. Embeddings & Index | Encode chunks with MiniLM-L6-v2 and store in FAISS |
| 5. Retrieval | Cosine-similarity search with parent-context promotion |
| 6. Generation | Grounded Q&A with GPT-4o |
| 7. Agentic RAG | Tool-calling agent that decides when and what to search |
| 8. Gradio UI | Chat interface powered by the agent |

**Requirements:** `pypdf`, `sentence-transformers`, `faiss-cpu`, `openai`, `gradio`  
**API key:** set `OPENAI_API_KEY` in your environment before running.


In [2]:
# Uncomment to install dependencies (run once)
# %pip install pypdf pymupdf sentence-transformers faiss-cpu gradio openai

import json
import logging
import os
import re
import warnings
from pathlib import Path

import faiss
import gradio as gr
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# Suppress Python warnings
warnings.filterwarnings("ignore")

# Suppress HuggingFace / SentenceTransformers logging warnings
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# Suppress the HuggingFace unauthenticated-request warning
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


'false'

## 1. Data Loading

Load the synthetic clinical notes PDF and extract its raw text.
Update `PDF_PATH` to point to your local copy of the file.

In [3]:
PDF_PATH = Path("data/synthetic_clinical_notes_for_rag.pdf")  # <-- update this path

reader = PdfReader(PDF_PATH)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Extracted {len(text):,} characters from {len(reader.pages)} pages.")
print(text[:500])


Extracted 39,115 characters from 24 pages.
Synthetic Clinical Notes Bundle (for NLP/RAG testing)
Document ID: SYNTH-CLIN-2026-03-03
Fictional data. Do not use for clinical care.
Page 1
 Synthetic Clinical Notes Bundle
 A multi-encounter, EHR-style document designed for NLP / LLM RAG experiments.
Created: March 03, 2026 | Purpose: Retrieval-Augmented Generation (RAG) testing, parsing, and
evaluation.
Important: All names, identifiers, facilities, and clinical details in this PDF are fictional and generated for
testing. Do not use for medi


## 2. Text Cleaning

Strip repeated page headers and normalise whitespace to produce clean, searchable text.

In [4]:
# Remove repeated page-header boilerplate
clean_text = re.sub(r"Synthetic Clinical Notes Bundle.*?Page \d+\n", "", text, flags=re.S)
clean_text = re.sub(r"\n{2,}", "\n", clean_text).strip()

# Top-level section titles that divide the document
SECTION_TITLES = [
    r"1\) Patient Snapshot",
    r"2\) Encounter A - Emergency Department",
    r"3\) Encounter A - Inpatient Admission",
    r"4\) Diagnostics: Radiology \+ Laboratory Results",
    r"5\) Hospital Course: Progress Notes, Nursing, RT, Pharmacy",
    r"6\) Medication Administration Record \(Excerpt\)",
    r"7\) Discharge Documentation",
    r"8\) Post-Discharge Follow-Up",
    r"9\) Appendix: Lab Timeline \+ Structured Variants",
]

section_pattern = re.compile(r"(?m)^(" + "|".join(SECTION_TITLES) + r")$")
section_matches = list(section_pattern.finditer(clean_text))
print(f"Found {len(section_matches)} top-level sections.")


Found 9 top-level sections.


## 3. Text Chunking

Splitting is done in three passes to balance retrieval precision with clinical context:

1. **Section chunking** – split on major document sections (Patient Snapshot, ED Note, Discharge, etc.) to preserve clinical structure.
2. **Sub-chunking** – detect internal note headers (HISTORY, PHYSICAL EXAM, LABS …) and split further, improving retrieval granularity.
3. **Merging** – adjacent sub-chunks within the same section are merged until they reach `MAX_CHARS`, preventing overly small fragments that hurt retrieval.

In [5]:
# Pass 1 – section-level chunks
chunks = []
for i, m in enumerate(section_matches):
    start = m.start()
    end = section_matches[i + 1].start() if i + 1 < len(section_matches) else len(clean_text)
    body = clean_text[start:end].strip()
    title = m.group(1).strip()
    body = re.sub(rf"^{re.escape(title)}\n?", "", body).strip()
    chunks.append({"title": title, "text": body})

print(f"Created {len(chunks)} section chunks.")
for c in chunks:
    print(f"  {c['title']} ({len(c['text'])} chars)")


Created 9 section chunks.
  1) Patient Snapshot (1164 chars)
  2) Encounter A - Emergency Department (3398 chars)
  3) Encounter A - Inpatient Admission (2964 chars)
  4) Diagnostics: Radiology + Laboratory Results (2954 chars)
  5) Hospital Course: Progress Notes, Nursing, RT, Pharmacy (9198 chars)
  6) Medication Administration Record (Excerpt) (3337 chars)
  7) Discharge Documentation (3805 chars)
  8) Post-Discharge Follow-Up (3952 chars)
  9) Appendix: Lab Timeline + Structured Variants (3388 chars)


In [6]:
# Pass 2 – sub-chunk on internal clinical note headers
SUBHEADER_RE = re.compile(
    r"(?m)^("
    r"[A-Z][A-Z0-9 &/\-\(\)\+\.:]{4,}|"
    r"Appendix [A-Z].+|"
    r"(?:CBC|CMP|BMP|ABG|ECG|A1c) \(.+\)"
    r")$"
)

subchunks = []
for chunk in chunks:
    body = chunk["text"]
    sub_matches = list(SUBHEADER_RE.finditer(body))

    if not sub_matches:
        # No sub-headers found – keep the whole section as one chunk
        subchunks.append({
            "parent_title": chunk["title"],
            "sub_title": chunk["title"],
            "text": body,
        })
        continue

    for i, m in enumerate(sub_matches):
        start = m.start()
        end = sub_matches[i + 1].start() if i + 1 < len(sub_matches) else len(body)
        sub_title = m.group(1).strip()
        sub_body = body[start:end].strip()
        sub_body = re.sub(rf"^{re.escape(sub_title)}\n?", "", sub_body).strip()
        subchunks.append({
            "parent_title": chunk["title"],
            "sub_title": sub_title,
            "text": sub_body,
        })

print(f"Created {len(subchunks)} sub-chunks.")


Created 86 sub-chunks.


In [7]:
# Pass 3 – merge small adjacent sub-chunks within the same section
MAX_CHARS = 1200  # maximum characters per final chunk
MIN_CHARS = 300   # chunks below this are merged with the previous one

merged_chunks = []
current = None

for s in subchunks:
    part = (
        s["text"]
        if s["sub_title"] == s["parent_title"]
        else f"{s['sub_title']}\n{s['text']}".strip()
    )
    new_section = current is None or s["parent_title"] != current["parent_title"]
    would_overflow = current and len(current["text"]) + len(part) > MAX_CHARS

    if new_section or would_overflow:
        if current:
            merged_chunks.append(current)
        current = {
            "parent_title": s["parent_title"],
            "sub_titles": [s["sub_title"]],
            "text": part,
        }
    else:
        current["sub_titles"].append(s["sub_title"])
        current["text"] += "\n\n" + part

if current:
    merged_chunks.append(current)

# Absorb any trailing micro-chunks into the preceding chunk
final_chunks = []
for c in merged_chunks:
    same_parent = final_chunks and c["parent_title"] == final_chunks[-1]["parent_title"]
    if same_parent and len(c["text"]) < MIN_CHARS:
        final_chunks[-1]["sub_titles"] += c["sub_titles"]
        final_chunks[-1]["text"] += "\n\n" + c["text"]
    else:
        final_chunks.append(c)

print(f"Final chunk count: {len(final_chunks)}")
for c in final_chunks[:5]:
    print(f"  [{c['parent_title']}] {c['sub_titles']} – {len(c['text'])} chars")


Final chunk count: 30
  [1) Patient Snapshot] ['1) Patient Snapshot'] – 1164 chars
  [2) Encounter A - Emergency Department] ['EMERGENCY DEPARTMENT NOTE', 'CHIEF COMPLAINT:', 'HISTORY OF PRESENT ILLNESS:', 'REVIEW OF SYSTEMS:', 'PAST MEDICAL HISTORY:'] – 1165 chars
  [2) Encounter A - Emergency Department] ['PAST SURGICAL HISTORY:', 'ALLERGIES:', 'SOCIAL HISTORY:', 'PHYSICAL EXAM:'] – 908 chars
  [2) Encounter A - Emergency Department] ['LABS:', 'IMAGING:', 'ED COURSE / MEDICAL DECISION MAKING:', 'IMPRESSION:', 'DISPOSITION:', 'CARDIOLOGY DIAGNOSTIC REPORT'] – 1335 chars
  [3) Encounter A - Inpatient Admission] ['INPATIENT HISTORY & PHYSICAL (H&P)', 'IDENTIFYING INFORMATION:', 'CHIEF COMPLAINT:', 'HISTORY OF PRESENT ILLNESS:', 'PAST MEDICAL HISTORY:', 'PAST SURGICAL HISTORY:'] – 1100 chars


## 4. Embeddings & FAISS Index

Each final chunk is encoded into a 384-dimensional dense vector using `sentence-transformers/all-MiniLM-L6-v2`. Vectors are stored in a FAISS `IndexFlatIP` (inner-product / cosine similarity, since embeddings are L2-normalised). The chunk metadata is saved alongside as a JSON file so the index and records stay in sync.

In [8]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_PATH = "clinical_chunks.index"
RECORDS_PATH = "clinical_chunks.json"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

records = [
    {"parent_title": c["parent_title"], "sub_titles": c["sub_titles"], "text": c["text"]}
    for c in final_chunks
]
texts = [r["text"] for r in records]

embeddings = embed_model.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

# Build and persist FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])  # cosine sim (vectors are L2-normalised)
index.add(embeddings)
faiss.write_index(index, INDEX_PATH)

with open(RECORDS_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Indexed {len(records)} chunks → saved to '{INDEX_PATH}' and '{RECORDS_PATH}'.")


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.64s/it]

Indexed 30 chunks → saved to 'clinical_chunks.index' and 'clinical_chunks.json'.


## 5. Retrieval

Load the persisted index and records, then define two retrieval helpers:

- **`retrieve_chunks`** – returns the top-*k* most similar chunks by cosine similarity.
- **`retrieve_parent_context`** – promotes results to parent-section level, returning all chunks that share the same top section for richer context.

In [9]:
# Load persisted index and chunk records
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
index = faiss.read_index(INDEX_PATH)

with open(RECORDS_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)


def retrieve_chunks(question: str, k: int = 3) -> list[dict]:
    """Return the top-k chunks most similar to *question*."""
    q_vec = embed_model.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    return [
        {
            "score": float(score),
            "parent_title": records[i]["parent_title"],
            "sub_titles": records[i]["sub_titles"],
            "text": records[i]["text"],
        }
        for score, i in zip(scores[0], ids[0])
    ]


def retrieve_parent_context(question: str, k: int = 3) -> list[dict]:
    """Return all chunks that share the top-scoring section for richer context."""
    top_hits = retrieve_chunks(question, k=k)
    if not top_hits:
        return []
    best_parent = top_hits[0]["parent_title"]
    return [
        {"parent_title": r["parent_title"], "sub_titles": r["sub_titles"], "text": r["text"]}
        for r in records
        if r["parent_title"] == best_parent
    ]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1249.12it/s]


In [10]:
# Quick smoke-test for the retrieval pipeline
test_question = "What was the patient's heart rate in the ER?"
hits = retrieve_parent_context(test_question)

print(f"Query: {test_question!r}\n")
for h in hits:
    print(f"  Section : {h['parent_title']}")
    print(f"  Sub-headers: {h['sub_titles']}")
    print(f"  Preview : {h['text'][:200].strip()}\n")


Query: "What was the patient's heart rate in the ER?"

  Section : 2) Encounter A - Emergency Department
  Sub-headers: ['EMERGENCY DEPARTMENT NOTE', 'CHIEF COMPLAINT:', 'HISTORY OF PRESENT ILLNESS:', 'REVIEW OF SYSTEMS:', 'PAST MEDICAL HISTORY:']
  Preview : EMERGENCY DEPARTMENT NOTE
Date/Time: 2026-02-26 13:45  |  Facility: Cityview Medical Center (synthetic)
Attending: Priya Natarajan, MD  |  Resident/APP: Jordan Li, PA-C
PATIENT: Jordan, Avery (Fiction

  Section : 2) Encounter A - Emergency Department
  Sub-headers: ['PAST SURGICAL HISTORY:', 'ALLERGIES:', 'SOCIAL HISTORY:', 'PHYSICAL EXAM:']
  Preview : PAST SURGICAL HISTORY:
- Appendectomy (2004)

ALLERGIES:
- Penicillin: "rash as a child" (history unclear)
- No known food allergies
HOME MEDICATIONS (per patient, may be incomplete):
- Albuterol HFA

  Section : 2) Encounter A - Emergency Department
  Sub-headers: ['LABS:', 'IMAGING:', 'ED COURSE / MEDICAL DECISION MAKING:', 'IMPRESSION:', 'DISPOSITION:', 'CARDIOLOGY DIAGNOSTIC R

### Generation: 
The system will feed your question along with the retrieved clinical context to an LLM (Large Language Model), ensuring the AI answers accurately based only on the provided clinical note.

In [11]:
# Set  OpenAI API key via an environment variable (never hard-code it)
# export OPENAI_API_KEY="sk-..."  ← set this in  shell or .env file
client = OpenAI()  # reads OPENAI_API_KEY from the environment automatically


In [12]:
SYSTEM_PROMPT = (
    "You answer only from the provided clinical note context. "
    "Be brief and precise. "
    "If the answer is not present, say: Not found in the provided clinical notes."
)


def generate_answer(question: str, k: int = 3, max_tokens: int = 300) -> str:
    """Retrieve relevant chunks and generate a grounded answer with GPT-4o."""
    hits = retrieve_parent_context(question, k=k)

    context = "\n\n".join(
        f"{h['parent_title']} | {' | '.join(h['sub_titles'])}\n{h['text']}"
        for h in hits
    )

    user_prompt = (
        f"Question: {question}\n\n"
        f"Clinical context:\n{context}\n\n"
        "Rules:\n"
        "- Answer only from the context above.\n"
        "- Do not guess or hallucinate.\n"
        "- If the answer is missing, say: Not found in the provided clinical notes.\n"
        "- Keep the answer short and exact."
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=0,  # deterministic for a RAG Q&A task
    )
    return response.choices[0].message.content


In [13]:
q = "What was the patient's heart rate in the ER?"
print(f"Q: {q}\nA: {generate_answer(q)}")


Q: What was the patient's heart rate in the ER?
A: The patient's heart rate in the ER was 112 bpm.


In [14]:
q = "What are Avery's active medications?"
print(f"Q: {q}\nA: {generate_answer(q)}")


Q: What are Avery's active medications?
A: Metformin ER 1000 mg, Lisinopril 10 mg, Atorvastatin 20 mg, Sertraline 50 mg, Fluticasone/Salmeterol 250/50, Albuterol HFA.


In [15]:
q = "What was the patient's discharge diagnosis?"
print(f"Q: {q}\nA: {generate_answer(q)}")


Q: What was the patient's discharge diagnosis?
A: The patient's discharge diagnosis was community-acquired pneumonia (right lower lobe).


## 7. Agentic RAG

The baseline pipeline uses *fixed* retrieval: every question triggers the same search-then-generate flow.  
The **agent** adds tool-use and decision-making: the model receives a `search_notes` tool and decides *when* to call it and *what* to search for before producing a final answer.  

The loop follows the standard two-turn Chat Completions tool-call pattern:

1. First call → model emits a `tool_calls` block.
2. Tool is executed locally (calls `retrieve_parent_context`).
3. Second call → model reads tool output and returns the final answer.

In [16]:
AGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_notes",
            "description": (
                "Search the clinical notes for information relevant to the given question "
                "and return the most relevant text chunks."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The clinical question to search for.",
                    },
                    "k": {
                        "type": "integer",
                        "description": "Number of top chunks to retrieve (default 3).",
                    },
                },
                "required": ["question"],
                "additionalProperties": False,
            },
        },
    }
]

AGENT_SYSTEM_PROMPT = (
    "You are a clinical note assistant. "
    "Always call the search_notes tool before answering. "
    "Answer only from the tool results – do not use prior knowledge. "
    'If the answer is absent, say: "Not found in the provided clinical notes." '
    "Keep answers short and exact."
)


def search_notes(question: str, k: int = 3) -> list[dict]:
    """Tool implementation: retrieve clinical note chunks for *question*."""
    return [
        {
            "parent_title": h["parent_title"],
            "sub_titles": h["sub_titles"],
            "text": h["text"][:1200],
        }
        for h in retrieve_parent_context(question, k=k)
    ]


def agent_answer(question: str, model_name: str = "gpt-4o-mini", max_tokens: int = 300) -> str:
    """
    Answer *question* using an agentic RAG loop.

    The model is given the `search_notes` tool and must call it before
    producing a final answer (tool_choice='required').
    """
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # Turn 1 – model decides what to search
    first_response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=AGENT_TOOLS,
        tool_choice="required",
        max_tokens=max_tokens,
        temperature=0,
    )

    assistant_msg = first_response.choices[0].message
    tool_calls = assistant_msg.tool_calls

    if not tool_calls:
        return assistant_msg.content or "No tool call returned."

    # Turn 1.5 – execute each tool call locally
    messages.append(assistant_msg)
    for call in tool_calls:
        args = json.loads(call.function.arguments)
        result = search_notes(args["question"], args.get("k", 3))
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result, ensure_ascii=False),
        })

    # Turn 2 – model reads tool output and produces the final answer
    second_response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
    )
    return second_response.choices[0].message.content or "No final text returned."


In [17]:
# Smoke-test the agent
q = "What was the patient's discharge diagnosis?"
print(f"Q: {q}\nA: {agent_answer(q)}")


Q: What was the patient's discharge diagnosis?
A: The patient's discharge diagnosis was community-acquired pneumonia (right lower lobe), with secondary diagnoses of asthma exacerbation, type 2 diabetes with hyperglycemia, hypertension, and depression/anxiety.


In [18]:
def chat_fn(message: str, history: list) -> str:
    """Gradio callback – route each user message through the agent."""
    return agent_answer(message)


demo = gr.ChatInterface(
    fn=chat_fn,
    title="Clinical Notes RAG Assistant",
    description="Ask questions about the synthetic patient record (Avery Jordan).",
    examples=[
        "What was the patient's heart rate in the ER?",
        "What are Avery's active medications?",
        "What was the discharge diagnosis?",
        "What antibiotics was the patient given?",
    ],
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


---
*All patient data in this notebook is entirely synthetic and generated for NLP/RAG testing purposes only. Do not use for clinical decision-making.*